In [1]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 30.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [2]:
import pandas as pd
import numpy as np
import torch
import transformers
from bertopic import BERTopic
import os
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
from transformers import AutoTokenizer, pipeline
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import TextGeneration
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import openai
import lightgbm as lgb
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import PCA
import nltk
import re
import plotly.io as pio
nltk.download('punkt_tab')
np.random.seed(42)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create Degendered Dataset for Topic Modeling

In [5]:
# os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
# df1 = pd.read_csv('../data/letters_2021.csv')
# df2 = pd.read_csv('../data/sentence_sets_trimmed.csv', encoding='mac-roman')
# df2 = df2[df2['full_text_tokens'] > 10]

## Degender Data

In [6]:
# raw_pairs = {}
# with open("../data/gendered_term_list.txt", "r", encoding="utf-8") as f:
#     for line in f:
#         key, value = line.strip().split("\t")
#         raw_pairs[key] = value

In [7]:
# degender_mapping = {
#     rf"(?:^|\b|[^\w\s])(?P<token>{re.escape(k)})(?P<suffix>'s|’s)?(?=\b|[^\w\s]|$)": v
#     for k, v in raw_pairs.items()
# }

### First Dataset

In [8]:
# df1 = df1[['LETTERTEXT', 'LETTER_GENDER']]
# df1 = df1.rename(columns={'LETTERTEXT':'full_text', 'LETTER_GENDER':'label'})
# df1['full_text'] = df1['full_text'].str.lower()

In [9]:
# def apply_degendering(text):
#     for pattern, replacement in degender_mapping.items():
#         def repl(match):
#             token = match.group("token")
#             suffix = match.group("suffix") or ""
#             replacement_base = raw_pairs.get(token.lower(), token)
#             return match.group(0).replace(token + suffix, replacement_base + suffix)
#         text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
#     return text

In [10]:
# df1['full_text'] = df1['full_text'].apply(apply_degendering)

### Second Dataset

In [11]:
# df2 = df2[['TEXT', 'applicant_gender']]
# df2 = df2.rename(columns={'applicant_gender':'label', 'TEXT':'full_text'})
# df2['full_text'] = df2['full_text'].str.lower()

In [12]:
# df2['full_text'] = df2['full_text'].apply(apply_degendering)

### Combine Datasets

In [13]:
# df = pd.concat([df1, df2], ignore_index=True)

In [14]:
# gender_label_mapping = {
#     'F':0,
#     'female':0,
#     'M':1,
#     'male':1
# }

In [15]:
# df['label'] = df['label'].replace(gender_label_mapping)

In [16]:
# df.to_csv('../data/combined_letters_degendered_topic_modeling.csv', index=False)

# Read Data

In [17]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered_topic_modeling.csv')

# Process Data

In [18]:
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42, stratify=train_val_df['label'])

In [19]:
letters = train_df['full_text']
sentences = [sent_tokenize(letter) for letter in letters]
sentences = pd.concat([train_df.reset_index(), pd.Series(sentences, name='sentences')], ignore_index=True, axis=1)
sentences = sentences.set_index(0)
sentences = sentences.rename(columns={1:'full_text', 2:'label', 3:'sentences'})
sentences.index.name = None
sentences = sentences[['sentences']].explode('sentences')
train_df_sentences = pd.merge(train_df, sentences, left_index=True, right_index=True)

In [20]:
letters = val_df['full_text']
sentences_val = [sent_tokenize(letter) for letter in letters]
sentences_val = pd.concat([val_df.reset_index(), pd.Series(sentences_val, name='sentences')], ignore_index=True, axis=1)
sentences_val = sentences_val.set_index(0)
sentences_val = sentences_val.rename(columns={1:'full_text', 2:'label', 3:'sentences'})
sentences_val.index.name = None
sentences_val = sentences_val[['sentences']].explode('sentences')
val_df_sentences = pd.merge(val_df, sentences_val, left_index=True, right_index=True)

In [21]:
letters = test_df['full_text']
sentences_test = [sent_tokenize(letter) for letter in letters]
sentences_test = pd.concat([test_df.reset_index(), pd.Series(sentences_test, name='sentences')], ignore_index=True, axis=1)
sentences_test = sentences_test.set_index(0)
sentences_test = sentences_test.rename(columns={1:'full_text', 2:'label', 3:'sentences'})
sentences_test.index.name = None
sentences_test = sentences_test[['sentences']].explode('sentences')
test_df_sentences = pd.merge(test_df, sentences_test, left_index=True, right_index=True)

### Remove Name Tokens

In [30]:
# remove_name_tokens = {
#     r'(?:^|\b|[^\w\s]+)identifier(?:\b|[^\w\s]+|$)': '',
#     r'(?:^|\b|[^\w\s]+)possible_identifier(?:\b|[^\w\s]+|$)': '',
#     r'(?:^|\b|[^\w\s]+)first_name(?:\b|[^\w\s]+|$)': '',
#     r'(?:^|\b|[^\w\s]+)middle_name(?:\b|[^\w\s]+|$)': '',
#     r'(?:^|\b|[^\w\s]+)last_name(?:\b|[^\w\s]+|$)': '',
# }

In [31]:
remove_name_tokens = {
    r'(?:^|\b|[^\w\s]+)first_name(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)middle_name(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)last_name(?:\b|[^\w\s]+|$)': ' identifier ',
}

In [32]:
train_df_sentences['sentences'] = train_df_sentences['sentences'].replace(remove_name_tokens, regex=True)

In [33]:
val_df_sentences['sentences'] = val_df_sentences['sentences'].replace(remove_name_tokens, regex=True)

In [34]:
test_df_sentences['sentences'] = test_df_sentences['sentences'].replace(remove_name_tokens, regex=True)

# Topic Modeling

In [35]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(train_df_sentences['sentences'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/3861 [00:00<?, ?it/s]

In [36]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [37]:
# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm", top_n_words=10)

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# GPT-3.5
client = openai.OpenAI(api_key="sk-...")
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""
openai_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    "POS": pos_model,
    # "zephyr": zephyr

}

In [38]:
topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True,
)

# Train model
topics, probs = topic_model.fit_transform(train_df_sentences['sentences'].tolist(), embeddings)



2025-04-22 17:37:18,601 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-22 17:41:32,954 - BERTopic - Dimensionality - Completed ✓
2025-04-22 17:41:32,959 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-22 17:42:01,483 - BERTopic - Cluster - Completed ✓
2025-04-22 17:42:01,540 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-22 17:42:22,836 - BERTopic - Representation - Completed ✓


# Save Model

In [39]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("../saved_models/topic_model_base/", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

# Analyze Topics

In [40]:
topic_info = topic_model.get_topic_info()

In [41]:
topic_info

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,51102,-1_research_surgery_possible_identifier_time,"[research, surgery, possible_identifier, time,...","[dr identifier, medical student, patients, ide...","[research, possible_identifier, staff, skills,...","[research, surgery, time, staff, room, skills,...","[an m3 im preceptor noted, “identifier is exce..."
1,0,7690,0_patients identifier_dr identifier_dr_care id...,"[patients identifier, dr identifier, dr, care ...","[identifier patients, patients identifier, pat...","[patients identifier, care identifier, patient...","[patient, physician, patient care, medical stu...",[identifier’s inclusive spirit is echoed in he...
2,1,4051,1_directress_professor_medical students_associate,"[directress, professor, medical students, asso...","[clinical faculty, anesthesiology university, ...","[directress, medical students, students reside...","[directress, professor, medical students, asso...",[i spent my early years on faculty at the univ...
3,2,3532,2_anesthesiologist_field anesthesiology_anesth...,"[anesthesiologist, field anesthesiology, anest...","[future anesthesiologist, anesthesiologist fut...","[field anesthesiology, career anesthesiology, ...","[anesthesiologist, anesthesiology, excellent a...",[she will make an outstanding anesthesiologist...
4,3,3334,3_icu_clerkship_medicine clerkship_internal me...,"[icu, clerkship, medicine clerkship, internal ...","[medicine clerkship, identifier surgical, inpa...","[icu, medicine clerkship, care unit, intensive...","[icu, clerkship, internal medicine, internal, ...",[i first came to know identifier during her fo...
...,...,...,...,...,...,...,...,...
92,91,167,91_pass_nbme_grade_grading,"[pass, nbme, grade, grading, pass fail, fail, ...","[honors pass, equivalent honors, grading, rece...","[grading, shelf exam, grade distribution, cler...","[pass, grade, shelf, honors, distribution, hig...",[these two week electives are pass / fail per ...
93,92,166,92_identifier summary_identifier reservations_...,"[identifier summary, identifier reservations, ...","[identifier summary, identifier great, summary...","[identifier summary, identifier reservations, ...","[highest recommendation, highest, reservations...","[in summary , identifier identifier has my..."
94,93,160,93_letter identifier_right letter_identifier w...,"[letter identifier, right letter, identifier w...","[letter waived, waived identifier, identifier ...","[letter identifier, right letter, identifier w...","[right, letter, customary, tu, identifier, , ,...",[identifier has waived her right to see this l...
95,94,152,94_reservation recommend_recommend reservation...,"[reservation recommend, recommend reservation,...","[recommend reservation, reservation recommend,...","[reservation recommend, recommend reservation,...","[reservation, highest endorsement, endorsement...",[i highly recommend her without any reservatio...


# Create Topics Dataframe

### Create Training Data

In [42]:
train_df_sentences['topic'] = topics
topics_df = train_df_sentences.copy()
topics_df['topic'] = topics_df['topic'].astype(str)
topics_df = topics_df.groupby(topics_df.index).agg({'topic':' '.join})
topics_df = pd.DataFrame(topics_df)
topics_df = topics_df['topic'].apply(lambda x:x.split(' '))
topics_df = pd.DataFrame(topics_df)
all_topics = set(cat for sublist in topics_df['topic'] for cat in sublist)
for topic in all_topics:
    topics_df[topic] = topics_df['topic'].apply(lambda x: 1 if topic in x else 0)
topics_df = topics_df.drop(columns=['topic'])
topics_df.columns = 'topic_' + topics_df.columns
train_df_sentences_collapsed = train_df_sentences[['full_text', 'label']].drop_duplicates()
topics_df = pd.merge(train_df_sentences_collapsed, topics_df, left_index=True, right_index=True)

### Create Validation Data

In [43]:
topics_val, probs_val = topic_model.transform(val_df_sentences['sentences'].tolist())

Batches:   0%|          | 0/431 [00:00<?, ?it/s]

2025-04-22 17:42:31,584 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-04-22 17:43:00,984 - BERTopic - Dimensionality - Completed ✓
2025-04-22 17:43:00,986 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-04-22 17:43:04,881 - BERTopic - Cluster - Completed ✓


In [44]:
val_df_sentences['topic'] = topics_val
val_topics_df = val_df_sentences.copy()
val_topics_df['topic'] = val_topics_df['topic'].astype(str)
val_topics_df = val_topics_df.groupby(val_topics_df.index).agg({'topic':' '.join})
val_topics_df = pd.DataFrame(val_topics_df)
val_topics_df = val_topics_df['topic'].apply(lambda x:x.split(' '))
val_topics_df = pd.DataFrame(val_topics_df)
all_topics = set(cat for sublist in val_topics_df['topic'] for cat in sublist)
for topic in all_topics:
    val_topics_df[topic] = val_topics_df['topic'].apply(lambda x: 1 if topic in x else 0)
val_topics_df = val_topics_df.drop(columns=['topic'])
val_topics_df.columns = 'topic_' + val_topics_df.columns
val_df_sentences_collapsed = val_df_sentences[['full_text', 'label']].drop_duplicates()
val_topics_df = pd.merge(val_df_sentences_collapsed, val_topics_df, left_index=True, right_index=True)

### Create Test Data

In [45]:
topics_test, probs_test = topic_model.transform(test_df_sentences['sentences'].tolist())

Batches:   0%|          | 0/1054 [00:00<?, ?it/s]

2025-04-22 17:43:18,802 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-04-22 17:43:38,127 - BERTopic - Dimensionality - Completed ✓
2025-04-22 17:43:38,128 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-04-22 17:43:47,960 - BERTopic - Cluster - Completed ✓


In [46]:
test_df_sentences['topic'] = topics_test
test_topics_df = test_df_sentences.copy()
test_topics_df['topic'] = test_topics_df['topic'].astype(str)
test_topics_df = test_topics_df.groupby(test_topics_df.index).agg({'topic':' '.join})
test_topics_df = pd.DataFrame(test_topics_df)
test_topics_df = test_topics_df['topic'].apply(lambda x:x.split(' '))
test_topics_df = pd.DataFrame(test_topics_df)
all_topics = set(cat for sublist in test_topics_df['topic'] for cat in sublist)
for topic in all_topics:
    test_topics_df[topic] = test_topics_df['topic'].apply(lambda x: 1 if topic in x else 0)
test_topics_df = test_topics_df.drop(columns=['topic'])
test_topics_df.columns = 'topic_' + test_topics_df.columns
test_df_sentences_collapsed = test_df_sentences[['full_text', 'label']].drop_duplicates()
test_topics_df = pd.merge(test_df_sentences_collapsed, test_topics_df, left_index=True, right_index=True)

In [47]:
topics_df.to_csv('../data/combined_letters_degendered_with_topics_train.csv', index=False)

In [48]:
val_topics_df.to_csv('../data/combined_letters_degendered_with_topics_val.csv', index=False)

In [49]:
test_topics_df.to_csv('../data/combined_letters_degendered_with_topics_test.csv', index=False)

# Analyze Important Topics

In [50]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
fig = topic_model.visualize_documents(train_df_sentences['sentences'].tolist(), hide_annotations=True, reduced_embeddings=reduced_embeddings)

In [51]:
fig

Output hidden; open in https://colab.research.google.com to view.

In [52]:
pio.write_html(fig, file="../figures/topics_base_interactive_plot.html", auto_open=True)